## 1. Environment and GPU requirement


# UCSD-PTGBM RSI/ADC controlled input-ablation — submission-oriented training pipeline

This is a **surgical restructuring** of the v5 notebook. The SwinUNETR model, optimizer,
DiceCE training objective, ROI, augmentation family, pretrained-encoder strategy, and the
4/5/5/6-channel ablation design are retained. The changes are primarily about experimental
validity, reproducibility, evaluation, and provenance.

## Scientific scope

The fixed framework tests incremental imaging-input value:

- `baseline`: T1, T1ce, T2, FLAIR
- `adc_only`: structural MRI + ADC
- `rsi_only`: structural MRI + RSI cellularity
- `full_6ch`: structural MRI + RSI cellularity + ADC

This notebook does **not** assert architectural novelty, clinical superiority, or "first" status.

## Frozen analysis rules

1. `SPLIT_SEED` is independent of `RUN_SEED`.
2. All arms and seed replicates use the identical subject split.
3. Model selection uses **subject-level** validation Dice after longitudinal timepoints are
   aggregated within subject.
4. If prediction and ground truth are both empty for a class/timepoint, Dice is excluded (`NaN`).
   If ground truth is empty but prediction is non-empty, Dice is **0**.
5. The same Gaussian sliding-window inference implementation is used everywhere.
6. Hypothesis testing is excluded from this training notebook; `stats_and_scores.ipynb` must use
   the subject-level exports.
7. Foreground labels remain `Class 1`–`Class 4` until the released integer mapping is authoritatively
   confirmed.
8. The historical internal test split is treated as a **development holdout**, because its results
   have already been inspected during debugging/model development.


In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "The frozen training protocol requires a CUDA GPU. "
        "Enable a Kaggle GPU accelerator before continuing."
    )

print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
!pip install "monai[nibabel]==1.6.0" einops scipy pandas -q
print("Environment dependencies installed.")


## 2. Imports, frozen configuration, and reproducibility


In [ ]:
import os
import sys
import re
import json
import random
import urllib.request
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import ndimage
from tqdm import tqdm
import einops

import torch
from torch.utils.data import Dataset

import monai
from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss
from monai.utils import set_determinism
from monai.inferers import sliding_window_inference
from monai.data import DataLoader, list_data_collate
from monai.transforms import (
    Compose,
    SpatialPadd,
    RandCropByLabelClassesd,
    RandFlipd,
    RandRotate90d,
    RandShiftIntensityd,
    EnsureTyped,
)

class ThesisConfig:
    VALID_MODES = ("baseline", "adc_only", "rsi_only", "full_6ch")

    # Experimental axes
    EXPERIMENT_MODE = "rsi_only"  # edit only this and RUN_SEED for planned runs
    RUN_SEED = 44
    SPLIT_SEED = 42               # FROZEN across all arms/seeds

    # Kaggle paths
    DATA_ROOT_RSI = Path(
        "/kaggle/input/datasets/iftekhar994/clean-brain-tumor-dataset/preprocessed_rsi_5class"
    )
    DATA_ROOT_BASE = Path(
        "/kaggle/input/datasets/iftekhar994/clean-brain-tumor-dataset/preprocessed_multimodal_5class"
    )
    WORKING_DIR = Path("/kaggle/working")
    PROJECT_DIR = WORKING_DIR / "ucsd_ptgbm_rsi_ablation_v6"

    # Frozen preprocessing-channel contract.
    # This is a declaration that must agree with the preprocessing pipeline; it is not inferred visually.
    BASE_CHANNEL_ORDER = ("T1", "T1ce", "T2", "FLAIR")
    RSI_CHANNEL_ORDER = ("T1", "T1ce", "T2", "FLAIR", "RSI_Cellularity", "ADC")
    CHANNEL_INDICES = {
        "baseline": (0, 1, 2, 3),
        "adc_only": (0, 1, 2, 3, 5),
        "rsi_only": (0, 1, 2, 3, 4),
        "full_6ch": (0, 1, 2, 3, 4, 5),
    }

    ROI_SIZE = (96, 96, 96)
    FEATURE_SIZE = 48
    NUM_CLASSES = 5
    CROP_RATIOS = (1, 3, 3, 3, 3)

    BATCH_SIZE = 1
    NUM_SAMPLES_PER_VOLUME = 2
    GRADIENT_ACCUMULATION_STEPS = 4
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    VAL_EVERY = 1
    EARLY_STOP_PATIENCE = 30
    MAX_EPOCHS = 300
    NUM_WORKERS = 2

    SLIDING_WINDOW_OVERLAP = 0.5
    SLIDING_WINDOW_MODE = "gaussian"

    USE_PRETRAINED = True
    PRETRAINED_URL = (
        "https://github.com/Project-MONAI/MONAI-extra-test-data/"
        "releases/download/0.8.1/model_swinvit.pt"
    )

    # Optional hard locks after the first frozen run.
    EXPECTED_SPLIT_SHA256 = None
    EXPECTED_PRETRAINED_SHA256 = None

    FRAGMENTATION_SAMPLE_N = 15
    FAIL_ON_FRAGMENTATION_WARNING = False

    def __init__(self):
        if self.EXPERIMENT_MODE not in self.VALID_MODES:
            raise ValueError(
                f"EXPERIMENT_MODE={self.EXPERIMENT_MODE!r} is invalid; "
                f"choose one of {self.VALID_MODES}."
            )
        self.CHANNEL_SELECTION = self.CHANNEL_INDICES[self.EXPERIMENT_MODE]
        self.IN_CHANNELS = len(self.CHANNEL_SELECTION)
        self.DATA_ROOT = (
            self.DATA_ROOT_BASE if self.EXPERIMENT_MODE == "baseline" else self.DATA_ROOT_RSI
        )
        self.ARM_NAME = f"{self.EXPERIMENT_MODE}_seed{self.RUN_SEED}"
        self.SPLIT_MANIFEST_PATH = (
            self.PROJECT_DIR / f"frozen_subject_split_seed{self.SPLIT_SEED}.json"
        )
        self.CHECKPOINT_DIR = self.PROJECT_DIR / "checkpoints" / self.ARM_NAME
        self.RESULTS_DIR = self.PROJECT_DIR / "results"
        self.ARM_RESULTS_DIR = self.RESULTS_DIR / self.ARM_NAME
        self.PRETRAINED_PATH = self.PROJECT_DIR / "pretrained" / "model_swinvit.pt"

        for path in (
            self.PROJECT_DIR,
            self.CHECKPOINT_DIR,
            self.RESULTS_DIR,
            self.ARM_RESULTS_DIR,
            self.PRETRAINED_PATH.parent,
        ):
            path.mkdir(parents=True, exist_ok=True)

config = ThesisConfig()

set_determinism(seed=config.RUN_SEED)
random.seed(config.RUN_SEED)
np.random.seed(config.RUN_SEED)
torch.manual_seed(config.RUN_SEED)
torch.cuda.manual_seed_all(config.RUN_SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

device = torch.device("cuda")

ENVIRONMENT = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "monai": monai.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "einops": getattr(einops, "__version__", "unknown"),
    "gpu": torch.cuda.get_device_name(0),
}

print(f"Arm: {config.ARM_NAME}")
print(f"RUN_SEED={config.RUN_SEED} | frozen SPLIT_SEED={config.SPLIT_SEED}")
print(f"Selected channels={config.CHANNEL_SELECTION} | n={config.IN_CHANNELS}")
print(json.dumps(ENVIRONMENT, indent=2))


## 3. Dataset manifest and immutable subject split


In [ ]:
def sha256_json(obj):
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def subject_of(timepoint_id):
    match = re.fullmatch(r"(.+)_([0-9]+)", timepoint_id)
    if match is None:
        raise ValueError(
            f"{timepoint_id!r} does not match required '<subject>_<numeric-timepoint>' pattern."
        )
    return match.group(1)

def file_ids(root):
    if not root.exists():
        raise FileNotFoundError(f"Dataset root does not exist: {root}")
    ids = sorted(p.stem for p in root.glob("*.npz"))
    if not ids:
        raise RuntimeError(f"No .npz files found under {root}")
    return ids

base_ids = file_ids(config.DATA_ROOT_BASE)
rsi_ids = file_ids(config.DATA_ROOT_RSI)

if set(base_ids) != set(rsi_ids):
    only_base = sorted(set(base_ids) - set(rsi_ids))
    only_rsi = sorted(set(rsi_ids) - set(base_ids))
    raise RuntimeError(
        "The two preprocessing roots do not contain identical timepoint IDs. "
        f"Only baseline (first 10): {only_base[:10]}; "
        f"only RSI (first 10): {only_rsi[:10]}"
    )

ALL_TIMEPOINT_IDS = sorted(base_ids)
ALL_SUBJECT_IDS = sorted({subject_of(t) for t in ALL_TIMEPOINT_IDS})
DATASET_ID_SHA256 = sha256_json(ALL_TIMEPOINT_IDS)

def generate_subject_level_split(timepoint_ids, split_seed, train_ratio=0.70, val_ratio=0.15):
    subject_to_timepoints = {}
    for tid in timepoint_ids:
        subject_to_timepoints.setdefault(subject_of(tid), []).append(tid)

    subjects = sorted(subject_to_timepoints)
    rng = random.Random(split_seed)
    rng.shuffle(subjects)

    n = len(subjects)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    parts = {
        "train": subjects[:train_end],
        "val": subjects[train_end:val_end],
        "test": subjects[val_end:],
    }
    return {
        key: sorted(t for s in subs for t in subject_to_timepoints[s])
        for key, subs in parts.items()
    }

def validate_split(split, all_timepoint_ids):
    required = {"train", "val", "test"}
    if set(split) != required:
        raise RuntimeError(f"Split keys must be exactly {required}; got {set(split)}")

    timepoint_sets = {k: set(split[k]) for k in required}
    if timepoint_sets["train"] & timepoint_sets["val"]:
        raise RuntimeError("Train/validation timepoint leakage detected.")
    if timepoint_sets["train"] & timepoint_sets["test"]:
        raise RuntimeError("Train/test timepoint leakage detected.")
    if timepoint_sets["val"] & timepoint_sets["test"]:
        raise RuntimeError("Validation/test timepoint leakage detected.")
    if set().union(*timepoint_sets.values()) != set(all_timepoint_ids):
        raise RuntimeError("Split does not cover exactly all dataset timepoints.")

    subject_sets = {k: {subject_of(t) for t in split[k]} for k in required}
    if subject_sets["train"] & subject_sets["val"]:
        raise RuntimeError("Train/validation SUBJECT leakage detected.")
    if subject_sets["train"] & subject_sets["test"]:
        raise RuntimeError("Train/test SUBJECT leakage detected.")
    if subject_sets["val"] & subject_sets["test"]:
        raise RuntimeError("Validation/test SUBJECT leakage detected.")

deterministic_split = generate_subject_level_split(
    ALL_TIMEPOINT_IDS, split_seed=config.SPLIT_SEED
)

if config.SPLIT_MANIFEST_PATH.exists():
    with config.SPLIT_MANIFEST_PATH.open("r") as f:
        dataset_splits = json.load(f)
    if dataset_splits != deterministic_split:
        raise RuntimeError(
            "Existing split manifest differs from the deterministic frozen split. "
            "Refusing to continue."
        )
    print(f"Loaded frozen split: {config.SPLIT_MANIFEST_PATH}")
else:
    dataset_splits = deterministic_split
    with config.SPLIT_MANIFEST_PATH.open("w") as f:
        json.dump(dataset_splits, f, indent=2)
    print(f"Created frozen split: {config.SPLIT_MANIFEST_PATH}")

validate_split(dataset_splits, ALL_TIMEPOINT_IDS)
SPLIT_SHA256 = sha256_json(dataset_splits)

if config.EXPECTED_SPLIT_SHA256 is not None:
    if SPLIT_SHA256 != config.EXPECTED_SPLIT_SHA256:
        raise RuntimeError(
            f"Split SHA256 mismatch: expected {config.EXPECTED_SPLIT_SHA256}, "
            f"got {SPLIT_SHA256}"
        )

print(f"Dataset: {len(ALL_TIMEPOINT_IDS)} timepoints / {len(ALL_SUBJECT_IDS)} subjects")
print(f"Dataset-ID SHA256: {DATASET_ID_SHA256}")
print(f"Frozen split SHA256: {SPLIT_SHA256}")
for key in ("train", "val", "test"):
    tids = dataset_splits[key]
    print(
        f"{key:5s}: {len(tids):3d} timepoints / "
        f"{len({subject_of(t) for t in tids}):3d} subjects"
    )


## 4. Cross-arm integrity and development-only label QC


In [ ]:
def load_npz_arrays(path):
    with np.load(path, allow_pickle=False) as data:
        required = {"image", "label"}
        missing = required - set(data.files)
        if missing:
            raise RuntimeError(f"{path}: missing NPZ keys {sorted(missing)}")
        image = np.asarray(data["image"])
        label = np.asarray(data["label"])
    return image, label

def validate_raw_arrays(image, label, path, expected_channels):
    if image.ndim != 4:
        raise RuntimeError(f"{path}: image must be [C,X,Y,Z], got {image.shape}")
    if label.ndim != 4 or label.shape[0] != 1:
        raise RuntimeError(f"{path}: label must be [1,X,Y,Z], got {label.shape}")
    if image.shape[0] != expected_channels:
        raise RuntimeError(
            f"{path}: expected {expected_channels} channels, got {image.shape[0]}"
        )
    if tuple(image.shape[1:]) != tuple(label.shape[1:]):
        raise RuntimeError(
            f"{path}: image/label spatial mismatch {image.shape[1:]} vs {label.shape[1:]}"
        )
    if not np.isfinite(image).all():
        raise RuntimeError(f"{path}: image contains NaN/Inf.")
    if not np.isfinite(label).all():
        raise RuntimeError(f"{path}: label contains NaN/Inf.")
    if not np.array_equal(label, label.astype(np.int64)):
        raise RuntimeError(f"{path}: label contains non-integer values.")

    observed = set(np.unique(label).astype(int).tolist())
    allowed = set(range(config.NUM_CLASSES))
    if not observed.issubset(allowed):
        raise RuntimeError(f"{path}: invalid labels {sorted(observed - allowed)}")

print("Running exact cross-arm equivalence checks...")
label_hasher = hashlib.sha256()
structural_hasher = hashlib.sha256()
advanced_hasher = hashlib.sha256()
for tid in tqdm(ALL_TIMEPOINT_IDS, desc="Cross-arm integrity"):
    base_path = config.DATA_ROOT_BASE / f"{tid}.npz"
    rsi_path = config.DATA_ROOT_RSI / f"{tid}.npz"

    base_image, base_label = load_npz_arrays(base_path)
    rsi_image, rsi_label = load_npz_arrays(rsi_path)

    validate_raw_arrays(base_image, base_label, base_path, expected_channels=4)
    validate_raw_arrays(rsi_image, rsi_label, rsi_path, expected_channels=6)

    if not np.array_equal(base_label, rsi_label):
        raise RuntimeError(f"{tid}: labels differ between preprocessing roots.")

    # Dataset-content fingerprints are updated only after the pair passes basic validation.
    tid_bytes = tid.encode("utf-8")
    label_hasher.update(tid_bytes)
    label_hasher.update(np.ascontiguousarray(base_label).tobytes())
    structural_hasher.update(tid_bytes)
    structural_hasher.update(np.ascontiguousarray(base_image).tobytes())
    advanced_hasher.update(tid_bytes)
    advanced_hasher.update(np.ascontiguousarray(rsi_image[4:]).tobytes())
    if not np.array_equal(base_image, rsi_image[:4]):
        max_abs = float(
            np.max(
                np.abs(
                    base_image.astype(np.float64) -
                    rsi_image[:4].astype(np.float64)
                )
            )
        )
        raise RuntimeError(
            f"{tid}: structural channels differ between roots; max absolute diff={max_abs}"
        )

LABEL_CONTENT_SHA256 = label_hasher.hexdigest()
STRUCTURAL_CONTENT_SHA256 = structural_hasher.hexdigest()
ADVANCED_CONTENT_SHA256 = advanced_hasher.hexdigest()

print("PASS: labels and structural channels are voxelwise identical across preprocessing roots.")
print(f"Label-content SHA256:      {LABEL_CONTENT_SHA256}")
print(f"Structural-content SHA256: {STRUCTURAL_CONTENT_SHA256}")
print(f"RSI/ADC-content SHA256:    {ADVANCED_CONTENT_SHA256}")

def analyze_label_fragmentation(timepoint_ids, data_root, n_sample, seed):
    rng = random.Random(seed)
    sample = rng.sample(timepoint_ids, min(n_sample, len(timepoint_ids)))
    connectivity_26 = ndimage.generate_binary_structure(rank=3, connectivity=3)
    rows = []

    for tid in sample:
        _, label = load_npz_arrays(data_root / f"{tid}.npz")
        label3d = label[0]
        for c in range(1, config.NUM_CLASSES):
            mask = label3d == c
            if not mask.any():
                continue
            cc, n_components = ndimage.label(mask, structure=connectivity_26)
            sizes = ndimage.sum(
                mask, cc, index=np.arange(1, n_components + 1)
            )
            rows.append(
                {
                    "timepoint_id": tid,
                    "class": c,
                    "voxels": int(mask.sum()),
                    "components_26conn": int(n_components),
                    "median_component_size": float(np.median(sizes)),
                }
            )
    return pd.DataFrame(rows)

# Do not use internal-test label morphology for development QC.
development_ids = sorted(dataset_splits["train"] + dataset_splits["val"])
frag_df = analyze_label_fragmentation(
    development_ids,
    config.DATA_ROOT,
    n_sample=config.FRAGMENTATION_SAMPLE_N,
    seed=config.SPLIT_SEED,
)

frag_summary = (
    frag_df.groupby("class")
    .agg(
        mean_components_26conn=("components_26conn", "mean"),
        mean_median_component_size=("median_component_size", "mean"),
        n_timepoints_with_class=("timepoint_id", "nunique"),
    )
    .reset_index()
)
print("\nDevelopment-only fragmentation diagnostic (26-connectivity):")
print(frag_summary.to_string(index=False))

suspicious = frag_summary[
    (frag_summary["mean_components_26conn"] > 20)
    & (frag_summary["mean_median_component_size"] < 3)
]

if len(suspicious):
    message = (
        "Fragmentation warning: historical 'many tiny components' pattern detected. "
        "Review preprocessing before training."
    )
    print("\nWARNING:", message)
    print(suspicious.to_string(index=False))
    if config.FAIL_ON_FRAGMENTATION_WARNING:
        raise RuntimeError(message)
else:
    print("\nNo sampled development class shows the historical fragmentation signature.")


## 5. Dataset and transforms


In [ ]:
class ThesisNpzDataset(Dataset):
    def __init__(self, timepoint_ids, data_root, transforms=None):
        self.timepoint_ids = list(timepoint_ids)
        self.data_root = Path(data_root)
        self.transforms = transforms

    def __len__(self):
        return len(self.timepoint_ids)

    def __getitem__(self, idx):
        tid = self.timepoint_ids[idx]
        path = self.data_root / f"{tid}.npz"
        image, label = load_npz_arrays(path)

        image = image.astype(np.float32, copy=False)
        label = label.astype(np.int64, copy=False)
        image = image[list(config.CHANNEL_SELECTION)]

        if image.shape[0] != config.IN_CHANNELS:
            raise RuntimeError(
                f"{path}: selected {image.shape[0]} channels; expected {config.IN_CHANNELS}"
            )

        sample = {
            "image": image,
            "label": label,
            "timepoint_id": tid,
            "subject_id": subject_of(tid),
        }
        return self.transforms(sample) if self.transforms is not None else sample

train_transforms = Compose(
    [
        EnsureTyped(keys=["image", "label"], data_type="tensor", track_meta=False),
        SpatialPadd(
            keys=["image", "label"],
            spatial_size=config.ROI_SIZE,
            mode=["constant", "constant"],
            constant_values=[0, 0],
        ),
        RandCropByLabelClassesd(
            keys=["image", "label"],
            label_key="label",
            spatial_size=config.ROI_SIZE,
            ratios=config.CROP_RATIOS,
            num_classes=config.NUM_CLASSES,
            num_samples=config.NUM_SAMPLES_PER_VOLUME,
        ),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
        RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
        RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
    ]
)

eval_transforms = Compose(
    [
        EnsureTyped(keys=["image", "label"], data_type="tensor", track_meta=False),
        SpatialPadd(
            keys=["image", "label"],
            spatial_size=config.ROI_SIZE,
            mode=["constant", "constant"],
            constant_values=[0, 0],
        ),
    ]
)

print("Dataset and frozen transform families defined.")


## 6. Training-only class weights and deterministic loaders


In [ ]:
def compute_inverse_frequency_class_weights(data_root, train_timepoint_ids):
    counts = np.zeros(config.NUM_CLASSES, dtype=np.int64)

    for tid in tqdm(train_timepoint_ids, desc="Training-label histogram"):
        _, label = load_npz_arrays(data_root / f"{tid}.npz")
        unique, unique_counts = np.unique(label, return_counts=True)
        for cls, count in zip(unique, unique_counts):
            cls = int(cls)
            if 0 <= cls < config.NUM_CLASSES:
                counts[cls] += int(count)

    if np.any(counts == 0):
        raise RuntimeError(f"At least one training class has zero voxels: {counts.tolist()}")

    total = counts.sum()
    weights = total / (config.NUM_CLASSES * counts.astype(np.float64))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32), counts

weights_cache = (
    config.PROJECT_DIR /
    f"class_weights_split-{SPLIT_SHA256[:12]}_labels-{LABEL_CONTENT_SHA256[:12]}_{config.NUM_CLASSES}class.pt"
)

if weights_cache.exists():
    cached = torch.load(weights_cache, map_location="cpu", weights_only=False)
    if cached.get("split_sha256") != SPLIT_SHA256:
        raise RuntimeError("Cached class weights were produced from a different split.")
    if cached.get("label_content_sha256") != LABEL_CONTENT_SHA256:
        raise RuntimeError("Cached class weights were produced from different label content.")
    class_weights = cached["weights"].to(torch.float32)
    class_counts = np.asarray(cached["counts"], dtype=np.int64)
    print(f"Loaded split-locked class weights: {weights_cache.name}")
else:
    class_weights, class_counts = compute_inverse_frequency_class_weights(
        config.DATA_ROOT, dataset_splits["train"]
    )
    torch.save(
        {
            "weights": class_weights,
            "counts": class_counts,
            "split_sha256": SPLIT_SHA256,
            "label_content_sha256": LABEL_CONTENT_SHA256,
        },
        weights_cache,
    )
    print(f"Saved split-locked class weights: {weights_cache.name}")

CLASS_WEIGHTS_SHA256 = hashlib.sha256(
    class_weights.cpu().numpy().tobytes()
).hexdigest()

print("Training voxel counts:", class_counts.tolist())
print("Mean-normalized class weights:", [round(float(x), 6) for x in class_weights])
print("Class-weight SHA256:", CLASS_WEIGHTS_SHA256)

train_ds = ThesisNpzDataset(
    dataset_splits["train"], config.DATA_ROOT, transforms=train_transforms
)
val_ds = ThesisNpzDataset(
    dataset_splits["val"], config.DATA_ROOT, transforms=eval_transforms
)
test_ds = ThesisNpzDataset(
    dataset_splits["test"], config.DATA_ROOT, transforms=eval_transforms
)

train_generator = torch.Generator()
train_generator.manual_seed(config.RUN_SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=True,
    collate_fn=list_data_collate,
    persistent_workers=False,
    drop_last=True,
    generator=train_generator,
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True,
    collate_fn=list_data_collate,
    persistent_workers=False,
)

test_loader = DataLoader(
    test_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=list_data_collate,
)

print(
    f"train={len(train_ds)} | val={len(val_ds)} | "
    f"internal_test={len(test_ds)} timepoints"
)


## 7. SwinUNETR and fail-closed pretrained loading


In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def load_inflated_pretrained_weights(model, weights_path, target_channels):
    checkpoint = torch.load(weights_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint.get("state_dict", checkpoint)

    fixed = {}
    for key, value in state_dict.items():
        new_key = key.replace("module.", "", 1) if key.startswith("module.") else key
        new_key = new_key.replace("mlp.fc1", "mlp.linear1").replace(
            "mlp.fc2", "mlp.linear2"
        )
        fixed[new_key] = value

    patch_key = "patch_embed.proj.weight"
    if patch_key not in fixed:
        raise RuntimeError(f"Pretrained checkpoint lacks {patch_key!r}.")

    w = fixed[patch_key]
    source_channels = int(w.shape[1])
    if source_channels > target_channels:
        raise RuntimeError(
            f"Pretrained checkpoint has {source_channels} input channels but model has "
            f"{target_channels}; channel reduction is not defined."
        )

    if source_channels != target_channels:
        extra = target_channels - source_channels
        pad = w.mean(dim=1, keepdim=True).repeat(1, extra, 1, 1, 1)
        fixed[patch_key] = torch.cat([w, pad], dim=1) * (
            source_channels / target_channels
        )
        print(
            f"Inflated patch embedding {source_channels}->{target_channels} "
            "by channel-mean repeat + magnitude rescaling."
        )

    msg = model.swinViT.load_state_dict(fixed, strict=False)

    if msg.missing_keys:
        raise RuntimeError(
            "Incomplete pretrained encoder load; missing keys: "
            + ", ".join(msg.missing_keys[:20])
        )

    allowed_prefixes = ("rotation_head", "contrastive_head", "convTrans3d", "norm.")
    invalid = [
        key for key in msg.unexpected_keys
        if not key.startswith(allowed_prefixes)
    ]
    if invalid or len(msg.unexpected_keys) > 8:
        raise RuntimeError(
            "Unexpected pretrained keys exceed known SSL-only heads. "
            f"all={msg.unexpected_keys}; invalid={invalid}"
        )

    print(
        f"Pretrained encoder verified: missing=0, "
        f"unexpected={len(msg.unexpected_keys)} known SSL-only keys."
    )
    return model

model = SwinUNETR(
    in_channels=config.IN_CHANNELS,
    out_channels=config.NUM_CLASSES,
    feature_size=config.FEATURE_SIZE,
    use_checkpoint=True,
    spatial_dims=3,
).to(device)

if not config.USE_PRETRAINED:
    raise RuntimeError(
        "The frozen primary study requires pretrained initialization. "
        "Random initialization must be a separately declared sensitivity experiment."
    )

if not config.PRETRAINED_PATH.exists():
    print("Downloading pretrained encoder checkpoint...")
    try:
        urllib.request.urlretrieve(config.PRETRAINED_URL, config.PRETRAINED_PATH)
    except Exception as exc:
        raise RuntimeError(
            "Pretrained download failed; refusing silent random initialization."
        ) from exc

PRETRAINED_SHA256 = sha256_file(config.PRETRAINED_PATH)
if config.EXPECTED_PRETRAINED_SHA256 is not None:
    if PRETRAINED_SHA256 != config.EXPECTED_PRETRAINED_SHA256:
        raise RuntimeError(
            f"Pretrained SHA256 mismatch: expected {config.EXPECTED_PRETRAINED_SHA256}, "
            f"got {PRETRAINED_SHA256}"
        )

model = load_inflated_pretrained_weights(
    model, config.PRETRAINED_PATH, config.IN_CHANNELS
)

print(f"Pretrained SHA256: {PRETRAINED_SHA256}")
print(f"SwinUNETR parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")


## 8. Loss, optimizer, scheduler, and AMP


In [ ]:
if len(class_weights) != config.NUM_CLASSES:
    raise RuntimeError(
        f"Expected {config.NUM_CLASSES} class weights, got {len(class_weights)}."
    )

loss_function = DiceCELoss(
    include_background=False,
    to_onehot_y=True,
    softmax=True,
    weight=class_weights.to(device),
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY,
)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=10
)
scaler = torch.amp.GradScaler("cuda", enabled=True)

print("Loss/optimizer/scheduler/AMP initialized.")


## 9. Complete run fingerprint and fail-closed checkpointing


In [ ]:
RUN_CONFIG = {
    "experiment_mode": config.EXPERIMENT_MODE,
    "arm_name": config.ARM_NAME,
    "run_seed": config.RUN_SEED,
    "split_seed": config.SPLIT_SEED,
    "split_sha256": SPLIT_SHA256,
    "dataset_id_sha256": DATASET_ID_SHA256,
    "label_content_sha256": LABEL_CONTENT_SHA256,
    "structural_content_sha256": STRUCTURAL_CONTENT_SHA256,
    "advanced_channel_content_sha256": ADVANCED_CONTENT_SHA256,
    "data_root_base": str(config.DATA_ROOT_BASE),
    "data_root_rsi": str(config.DATA_ROOT_RSI),
    "declared_base_channel_order": list(config.BASE_CHANNEL_ORDER),
    "declared_rsi_channel_order": list(config.RSI_CHANNEL_ORDER),
    "selected_channel_indices": list(config.CHANNEL_SELECTION),
    "in_channels": config.IN_CHANNELS,
    "num_classes": config.NUM_CLASSES,
    "architecture": "MONAI SwinUNETR",
    "feature_size": config.FEATURE_SIZE,
    "roi_size": list(config.ROI_SIZE),
    "crop_ratios": list(config.CROP_RATIOS),
    "batch_size": config.BATCH_SIZE,
    "num_samples_per_volume": config.NUM_SAMPLES_PER_VOLUME,
    "gradient_accumulation_steps": config.GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": config.LEARNING_RATE,
    "weight_decay": config.WEIGHT_DECAY,
    "val_every": config.VAL_EVERY,
    "early_stop_patience": config.EARLY_STOP_PATIENCE,
    "max_epochs": config.MAX_EPOCHS,
    "class_weights": [float(x) for x in class_weights],
    "class_weights_sha256": CLASS_WEIGHTS_SHA256,
    "pretrained_url": config.PRETRAINED_URL,
    "pretrained_sha256": PRETRAINED_SHA256,
    "sliding_window_overlap": config.SLIDING_WINDOW_OVERLAP,
    "sliding_window_mode": config.SLIDING_WINDOW_MODE,
    "absence_rule": (
        "both prediction and GT empty -> NaN/excluded; "
        "GT empty and prediction non-empty -> Dice 0"
    ),
    "checkpoint_selection": (
        "mean subject-level foreground Dice on validation subjects"
    ),
    "environment": ENVIRONMENT,
}

RUN_CONFIG_SHA256 = sha256_json(RUN_CONFIG)

metadata_path = config.ARM_RESULTS_DIR / "run_metadata.json"
with metadata_path.open("w") as f:
    json.dump(
        {"run_config_sha256": RUN_CONFIG_SHA256, "run_config": RUN_CONFIG},
        f,
        indent=2,
    )

print(f"Run-config SHA256: {RUN_CONFIG_SHA256}")
print(f"Run metadata: {metadata_path}")

LAST_CHECKPOINT = config.CHECKPOINT_DIR / "last_checkpoint.pt"
BEST_CHECKPOINT = config.CHECKPOINT_DIR / "best_checkpoint.pt"

def save_resilient_checkpoint(
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    best_metric,
    patience_counter,
    path,
):
    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_metric": float(best_metric),
        "patience_counter": int(patience_counter),
        "rng_state_cuda": torch.cuda.get_rng_state_all(),
        "rng_state_torch": torch.get_rng_state(),
        "rng_state_numpy": np.random.get_state(),
        "rng_state_python": random.getstate(),
        "train_generator_state": train_generator.get_state(),
        "run_config_sha256": RUN_CONFIG_SHA256,
        "split_sha256": SPLIT_SHA256,
    }
    torch.save(state, path)

    if Path(path) == BEST_CHECKPOINT:
        print(
            f"  Best checkpoint saved: {Path(path).name} | "
            f"sha256={sha256_file(path)[:16]}... | epoch={epoch}"
        )
    else:
        print(f"  Last checkpoint saved: {Path(path).name} | epoch={epoch}")

def load_resilient_checkpoint(path, model, optimizer, scheduler, scaler):
    path = Path(path)
    if not path.exists():
        return None

    checkpoint = torch.load(path, map_location=device, weights_only=False)
    if checkpoint.get("run_config_sha256") != RUN_CONFIG_SHA256:
        raise RuntimeError(
            "Checkpoint/run-config mismatch; refusing to resume. "
            f"checkpoint={checkpoint.get('run_config_sha256')}, "
            f"current={RUN_CONFIG_SHA256}"
        )
    if checkpoint.get("split_sha256") != SPLIT_SHA256:
        raise RuntimeError("Checkpoint was trained on a different subject split.")

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    torch.set_rng_state(checkpoint["rng_state_torch"].cpu())
    torch.cuda.set_rng_state_all(
        [state.cpu() for state in checkpoint["rng_state_cuda"]]
    )
    np.random.set_state(checkpoint["rng_state_numpy"])
    random.setstate(checkpoint["rng_state_python"])
    train_generator.set_state(checkpoint["train_generator_state"].cpu())

    print(f"Resumed verified checkpoint: {path}")
    return checkpoint

start_epoch = 0
best_val_subject_mean_dice = -1.0
patience_counter = 0

resume_state = load_resilient_checkpoint(
    LAST_CHECKPOINT, model, optimizer, lr_scheduler, scaler
)
if resume_state is not None:
    start_epoch = int(resume_state["epoch"]) + 1
    best_val_subject_mean_dice = float(resume_state["best_metric"])
    patience_counter = int(resume_state["patience_counter"])
    print(
        f"Resume at epoch={start_epoch}; historical best subject-level validation Dice="
        f"{best_val_subject_mean_dice:.4f}"
    )
else:
    print("No compatible v6 checkpoint found; starting a fresh run.")


## 10. Unified inference and explicit subject-level aggregation


In [ ]:
def _metadata_as_list(value, expected_length):
    if isinstance(value, str):
        result = [value]
    elif isinstance(value, (list, tuple)):
        result = [str(x) for x in value]
    elif isinstance(value, np.ndarray):
        result = [str(x) for x in value.tolist()]
    else:
        result = [str(value)]

    if len(result) != expected_length:
        raise RuntimeError(
            f"Metadata length mismatch: expected {expected_length}, got {len(result)}"
        )
    return result

def dice_with_explicit_empty_rule(pred_mask, gt_mask):
    pred_voxels = int(pred_mask.sum())
    gt_voxels = int(gt_mask.sum())

    if gt_voxels == 0 and pred_voxels == 0:
        return np.nan, "both_absent"
    if gt_voxels == 0 and pred_voxels > 0:
        return 0.0, "false_positive_empty_gt"

    intersection = int(np.logical_and(pred_mask, gt_mask).sum())
    dice = 2.0 * intersection / (pred_voxels + gt_voxels)
    return float(dice), "evaluable"

def aggregate_timepoints_to_subjects(timepoint_df):
    rows = []
    for (split_name, subject_id), group in timepoint_df.groupby(
        ["split", "subject_id"], sort=True
    ):
        row = {
            "split": split_name,
            "subject_id": subject_id,
            "n_timepoints": int(len(group)),
        }

        for c in range(config.NUM_CLASSES):
            values = group[f"dice_class_{c}"].to_numpy(dtype=float)
            finite = values[np.isfinite(values)]
            row[f"dice_class_{c}"] = (
                float(finite.mean()) if len(finite) else np.nan
            )
            row[f"evaluable_timepoints_class_{c}"] = int(len(finite))

        fg = np.asarray(
            [row[f"dice_class_{c}"] for c in range(1, config.NUM_CLASSES)],
            dtype=float,
        )
        finite_fg = fg[np.isfinite(fg)]
        row["foreground_mean_dice"] = (
            float(finite_fg.mean()) if len(finite_fg) else np.nan
        )
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values(["split", "subject_id"])
        .reset_index(drop=True)
    )

def summarize_subject_metrics(subject_df):
    fg = subject_df["foreground_mean_dice"].dropna()
    if len(fg) == 0:
        raise RuntimeError("No evaluable subject-level foreground Dice values.")

    summary = {
        "n_subjects": int(subject_df["subject_id"].nunique()),
        "subject_mean_foreground_dice": float(fg.mean()),
        "subject_median_foreground_dice": float(fg.median()),
    }
    for c in range(config.NUM_CLASSES):
        vals = subject_df[f"dice_class_{c}"].dropna()
        summary[f"subject_mean_dice_class_{c}"] = (
            float(vals.mean()) if len(vals) else np.nan
        )
    return summary

def evaluate_segmentation(
    model,
    loader,
    loss_fn,
    split_name,
    expected_timepoint_ids,
):
    # Single source of truth for validation and internal-test inference.
    model.eval()
    rows = []
    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=True):
                logits = sliding_window_inference(
                    inputs=images,
                    roi_size=config.ROI_SIZE,
                    sw_batch_size=1,
                    predictor=model,
                    overlap=config.SLIDING_WINDOW_OVERLAP,
                    mode=config.SLIDING_WINDOW_MODE,
                )
                loss = loss_fn(logits, labels)

            total_loss += float(loss.item())
            n_batches += 1

            pred = torch.argmax(logits, dim=1, keepdim=True)
            batch_size = int(pred.shape[0])
            timepoint_ids = _metadata_as_list(batch["timepoint_id"], batch_size)
            subject_ids = _metadata_as_list(batch["subject_id"], batch_size)

            for b in range(batch_size):
                pred_np = pred[b, 0].detach().cpu().numpy()
                gt_np = labels[b, 0].detach().cpu().numpy()

                row = {
                    "split": split_name,
                    "timepoint_id": timepoint_ids[b],
                    "subject_id": subject_ids[b],
                    "arm": config.ARM_NAME,
                    "run_seed": config.RUN_SEED,
                    "split_sha256": SPLIT_SHA256,
                    "run_config_sha256": RUN_CONFIG_SHA256,
                }

                for c in range(config.NUM_CLASSES):
                    pred_mask = pred_np == c
                    gt_mask = gt_np == c
                    value, status = dice_with_explicit_empty_rule(
                        pred_mask, gt_mask
                    )
                    row[f"dice_class_{c}"] = value
                    row[f"dice_status_class_{c}"] = status
                    row[f"pred_voxels_class_{c}"] = int(pred_mask.sum())
                    row[f"gt_voxels_class_{c}"] = int(gt_mask.sum())

                fg = np.asarray(
                    [
                        row[f"dice_class_{c}"]
                        for c in range(1, config.NUM_CLASSES)
                    ],
                    dtype=float,
                )
                finite_fg = fg[np.isfinite(fg)]
                row["foreground_mean_dice_timepoint"] = (
                    float(finite_fg.mean()) if len(finite_fg) else np.nan
                )
                rows.append(row)

    if n_batches == 0:
        raise RuntimeError(f"{split_name}: empty evaluation loader.")

    timepoint_df = (
        pd.DataFrame(rows)
        .sort_values(["subject_id", "timepoint_id"])
        .reset_index(drop=True)
    )

    if sorted(timepoint_df["timepoint_id"].tolist()) != sorted(expected_timepoint_ids):
        raise RuntimeError(
            f"{split_name}: evaluated IDs do not match the frozen split manifest."
        )

    subject_df = aggregate_timepoints_to_subjects(timepoint_df)
    summary = summarize_subject_metrics(subject_df)
    return total_loss / n_batches, timepoint_df, subject_df, summary

print(
    "Unified evaluator ready: Gaussian sliding-window inference, explicit empty-class handling, "
    "and within-subject longitudinal aggregation."
)


## 11. Training with subject-level validation checkpoint selection


In [ ]:
print("--- Training pipeline ---")
print(
    "Checkpoint metric: mean subject-level foreground Dice across validation subjects."
)

HISTORY_CSV = config.ARM_RESULTS_DIR / "training_history.csv"
if HISTORY_CSV.exists():
    history_rows = pd.read_csv(HISTORY_CSV).to_dict("records")
else:
    history_rows = []

for epoch in range(start_epoch, config.MAX_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    epoch_loss = 0.0
    n_steps = len(train_loader)
    progress = tqdm(
        train_loader, 
        desc=f"Epoch {epoch}/{config.MAX_EPOCHS - 1}", 
        mininterval=300.0  # Only updates the progress bar once every 5 minutes
    )

    for step_index, batch in enumerate(progress):
        inputs = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        group_start = (
            step_index // config.GRADIENT_ACCUMULATION_STEPS
        ) * config.GRADIENT_ACCUMULATION_STEPS
        current_group_size = min(
            config.GRADIENT_ACCUMULATION_STEPS,
            n_steps - group_start,
        )

        with torch.amp.autocast("cuda", enabled=True):
            logits = model(inputs)
            raw_loss = loss_function(logits, labels)
            scaled_loss = raw_loss / current_group_size

        scaler.scale(scaled_loss).backward()
        epoch_loss += float(raw_loss.item())

        is_group_end = (
            (step_index + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0
            or step_index + 1 == n_steps
        )
        if is_group_end:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        progress.set_postfix({"batch_loss": f"{raw_loss.item():.4f}"})

    mean_train_loss = epoch_loss / n_steps
    current_lr = float(optimizer.param_groups[0]["lr"])
    print(
        f"Epoch {epoch}: mean training loss={mean_train_loss:.4f} | "
        f"lr={current_lr:.8f}"
    )

    if (epoch + 1) % config.VAL_EVERY != 0:
        continue

    val_loss, _, val_subject_df, val_summary = evaluate_segmentation(
        model=model,
        loader=val_loader,
        loss_fn=loss_function,
        split_name="val",
        expected_timepoint_ids=dataset_splits["val"],
    )

    val_metric = val_summary["subject_mean_foreground_dice"]
    lr_scheduler.step(val_metric)

    print(f"Validation loss: {val_loss:.4f}")
    print(
        "Validation subject-level foreground Dice: "
        f"mean={val_summary['subject_mean_foreground_dice']:.4f}, "
        f"median={val_summary['subject_median_foreground_dice']:.4f}, "
        f"n={val_summary['n_subjects']}"
    )
    for c in range(config.NUM_CLASSES):
        print(
            f"  Class {c} subject-mean Dice: "
            f"{val_summary[f'subject_mean_dice_class_{c}']:.4f}"
        )

    history_row = {
        "epoch": epoch,
        "mean_train_loss": mean_train_loss,
        "val_loss": val_loss,
        "lr_before_scheduler_step": current_lr,
        **val_summary,
    }
    history_rows = [
        row for row in history_rows
        if int(row.get("epoch", -1)) != epoch
    ]
    history_rows.append(history_row)
    (
        pd.DataFrame(history_rows)
        .sort_values("epoch")
        .to_csv(HISTORY_CSV, index=False)
    )

    if val_metric > best_val_subject_mean_dice:
        best_val_subject_mean_dice = val_metric
        patience_counter = 0
        save_resilient_checkpoint(
            epoch,
            model,
            optimizer,
            lr_scheduler,
            scaler,
            best_val_subject_mean_dice,
            patience_counter,
            BEST_CHECKPOINT,
        )
        print("  [new best subject-level validation checkpoint]")
    else:
        patience_counter += 1
        print(
            "  [no improvement; "
            f"patience remaining={config.EARLY_STOP_PATIENCE - patience_counter}]"
        )

    save_resilient_checkpoint(
        epoch,
        model,
        optimizer,
        lr_scheduler,
        scaler,
        best_val_subject_mean_dice,
        patience_counter,
        LAST_CHECKPOINT,
    )

    if patience_counter >= config.EARLY_STOP_PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break


## 12. Standardized best-checkpoint export

This section exports **separate timepoint-level and subject-level files** using exactly the same
evaluator used for checkpoint selection.

The internal `test` split is preserved for continuity with the historical project, but its results
have already been inspected. Treat it as an **internal development holdout**, not untouched
confirmatory validation.

No hypothesis tests are calculated here.


In [ ]:
if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found: {BEST_CHECKPOINT}. Train this arm first."
    )

best_checkpoint = torch.load(
    BEST_CHECKPOINT, map_location=device, weights_only=False
)
if best_checkpoint.get("run_config_sha256") != RUN_CONFIG_SHA256:
    raise RuntimeError(
        "Best checkpoint does not match the current frozen run configuration."
    )

model.load_state_dict(best_checkpoint["model_state_dict"], strict=True)
print(f"Loaded best checkpoint from epoch {best_checkpoint['epoch']}.")

export_manifest = {}

for split_name, loader, manifest_key in (
    ("val", val_loader, "val"),
    ("internal_test", test_loader, "test"),
):
    mean_loss, timepoint_df, subject_df, summary = evaluate_segmentation(
        model=model,
        loader=loader,
        loss_fn=loss_function,
        split_name=split_name,
        expected_timepoint_ids=dataset_splits[manifest_key],
    )

    prefix = f"{split_name}_{config.ARM_NAME}_split-{SPLIT_SHA256[:12]}"
    timepoint_path = (
        config.ARM_RESULTS_DIR / f"{prefix}_timepoint_metrics.csv"
    )
    subject_path = (
        config.ARM_RESULTS_DIR / f"{prefix}_subject_metrics.csv"
    )

    timepoint_df.to_csv(timepoint_path, index=False)
    subject_df.to_csv(subject_path, index=False)

    export_manifest[split_name] = {
        "loss": float(mean_loss),
        "summary": summary,
        "timepoint_csv": str(timepoint_path),
        "timepoint_csv_sha256": sha256_file(timepoint_path),
        "subject_csv": str(subject_path),
        "subject_csv_sha256": sha256_file(subject_path),
    }

    print(f"\n=== {split_name.upper()} — SUBJECT LEVEL ===")
    print(json.dumps(summary, indent=2))
    print(f"Timepoint export: {timepoint_path.name}")
    print(f"Subject export:   {subject_path.name}")

manifest_path = config.ARM_RESULTS_DIR / "evaluation_export_manifest.json"
with manifest_path.open("w") as f:
    json.dump(
        {
            "arm": config.ARM_NAME,
            "run_config_sha256": RUN_CONFIG_SHA256,
            "split_sha256": SPLIT_SHA256,
            "best_checkpoint_epoch": int(best_checkpoint["epoch"]),
            "best_checkpoint_sha256": sha256_file(BEST_CHECKPOINT),
            "exports": export_manifest,
        },
        f,
        indent=2,
    )

print(f"\nEvaluation manifest: {manifest_path}")


## 13. Analysis boundary and remaining submission checks

### Statistics

Use only `*_subject_metrics.csv` in `stats_and_scores.ipynb` for paired inference.
Never treat `*_timepoint_metrics.csv` as independent patients.

The statistics notebook should explicitly lock:

- the primary baseline-vs-full subject-level endpoint;
- two-sided Wilcoxon settings and zero/tie handling;
- matched-pairs effect size and confidence intervals;
- Holm correction families;
- bootstrap method, **subject** resampling unit, and bootstrap seed.

### Locked confirmation set

The separate 42-subject / 59-timepoint BraTS challenge holdout should be evaluated only after this
pipeline and the statistical plan are frozen. Reuse the exact same inference and empty-class rules.
Use the wording **independent same-institution holdout** unless a stronger external-validation claim
is demonstrably justified.

### Preprocessing still requires a separate audit

This notebook verifies the arrays it receives, but it cannot prove registration, orientation,
physical spacing, intensity normalization, or label-resampling provenance. Archive and review the
preprocessing code/version before submission.

### Metrics beyond Dice

Do not add HD95/surface-distance claims until voxel spacing/affine provenance is available in an
auditable form. Once that is available, add physical-space boundary metrics with a pre-specified
empty-object convention.

### Label mapping

Do not infer clinical class identities from volume frequency or selected visual slices. Keep
`Class 1`–`Class 4` until an authoritative released-label mapping is obtained.
